![](img/logo_ucm.jpg)

# Ejercicio práctico-teórico: MLflow

En este ejercicio vas a consolidar los conceptos principales del tema 3: **tracking de experimentos**, **comparacion de runs**, **artefactos** y **carga de modelos desde MLflow**. 

Trabajaremos con el dataset `diabetes` de `scikit-learn` para evitar dependencias externas y poder centrarnos en el flujo de MLflow. El objetivo del ejercicio es afianzar conocimientos entrenando modelos y dejando evidencia reproducible de su flujo de construcción y evaluación. 

## Objetivos

1. Configurar un tracking local con MLflow.
2. Crear un experimento y registrar varios modelos como runs anidados.
3. Loggear parametros, metricas, tags, artefactos y el modelo entrenado.
4. Comparar los runs y seleccionar el mejor segun RMSE.
5. Cargar el mejor modelo desde una URI `runs:/...`.
6. Razonar las diferencias entre metricas, artefactos y trazabilidad.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Apartado 1: configuracion del experimento

Configura un tracking local para este ejercicio usando SQLite como backend de metadata, crea las carpetas necesarias y activa el experimento `tema3-mlflow`.

In [ ]:
EXPERIMENT_NAME = "tema3-mlflow"
TRACKING_DB = Path("mlflow_tarea_mlflow.db")
ARTIFACTS_DIR = Path("artifacts_tarea_mlflow")

ARTIFACTS_DIR.mkdir(exist_ok=True)

# TODO (1): configurar MLflow para que use TRACKING_DB con sqlite como tracking URI
# TODO (2): crear o activar el experimento EXPERIMENT_NAME

...

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


### Revisar los experimentos en la interfaz de MLflow

Una vez ejecutado el notebook, podemos levantar la interfaz web de MLflow para inspeccionar los runs, comparar metricas, revisar parametros y navegar por los artifacts registrados.

```bash
cd notebooks_clase/tema_3_mlflow
uv run mlflow ui --backend-store-uri sqlite:///mlflow_tarea_mlflow.db --port 5000
```

Despues, abre `http://127.0.0.1:5000` en el navegador.


## Apartado 2: carga y particion del dataset

Carga `diabetes`, separa `X` e `y` y haz una particion train/test con `test_size=0.2` y `random_state=42`.

In [ ]:
dataset = load_diabetes(as_frame=True)
df = dataset.frame.copy()

FEATURE_NAMES = dataset.feature_names
TARGET_NAME = "target"

# TODO (3): separar features y target
X = ...
y = ...

# TODO (4): hacer train_test_split
X_train, X_test, y_train, y_test = ...

print(df.head(3))
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## Apartado 3: helpers de evaluación

Usaremos tres candidatos sencillos de regresion y una funcion auxiliar para calcular métricas y generar un gráfico de residuos.

In [ ]:
def build_candidates():
    return {
        "linear_regression": Pipeline(
            steps=[("scaler", StandardScaler()), ("model", LinearRegression())]
        ),
        "random_forest": Pipeline(
            steps=[(
                "model",
                RandomForestRegressor(
                    n_estimators=200,
                    max_depth=6,
                    random_state=42,
                    n_jobs=-1,
                ),
            )]
        ),
        "gradient_boosting": Pipeline(
            steps=[("model", GradientBoostingRegressor(random_state=42))]
        ),
    }


def regression_metrics(y_true, y_pred):
    return {
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def make_residual_plot(y_true, y_pred, title, output_path):
    residuals = y_true - y_pred
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(y_pred, residuals, alpha=0.7)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Residuals - {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual")
    fig.tight_layout()
    fig.savefig(output_path, dpi=120)
    plt.close(fig)


## Apartado 4: entrenamiento y tracking con nested runs

Para cada candidato, entrena el pipeline, calcula métricas y registra en MLflow:

- parametros del estimador final
- metricas de test (`rmse`, `mae`, `r2`)
- tags útiles
- modelo en `artifact_path="model"`
- gráfico de residuos como artifact


In [ ]:
run_summaries = []

with mlflow.start_run(run_name="model-comparison-diabetes") as parent_run:
    mlflow.set_tag("exercise", "tema_3_mlflow")
    mlflow.set_tag("dataset", "diabetes")

    for candidate_name, pipeline in build_candidates().items():
        with mlflow.start_run(run_name=f"candidate-{candidate_name}", nested=True) as child_run:
            # TODO (5): entrenar el pipeline
            ...

            # TODO (6): predecir sobre X_test
            y_pred = ...
            metrics = regression_metrics(y_test, y_pred)

            final_model = pipeline.named_steps["model"]
            model_params = final_model.get_params()

            # TODO (7): loggear parametros del modelo y metricas
            ...

            # TODO (8): añadir tags utiles
            ...

            signature = infer_signature(X_train, pipeline.predict(X_train))

            # TODO (9): registrar el modelo con mlflow.sklearn.log_model(...)
            ...

            plot_path = ARTIFACTS_DIR / f"residuals_{candidate_name}.png"
            make_residual_plot(y_test, y_pred, candidate_name, plot_path)

            # TODO (10): registrar el grafico como artifact
            ...

            run_summaries.append(
                {
                    "candidate_name": candidate_name,
                    "run_id": child_run.info.run_id,
                    **metrics,
                }
            )

leaderboard = pd.DataFrame(run_summaries).sort_values("rmse").reset_index(drop=True)
leaderboard


## Apartado 5: seleccion del mejor run y carga del modelo

Selecciona el mejor run segun menor RMSE, construye una URI `runs:/.../model` y carga el modelo con `mlflow.pyfunc.load_model`.

In [ ]:
# TODO (11): seleccionar la mejor fila del leaderboard
best_row = ...

# TODO (12): construir la URI del modelo y cargarlo
best_model_uri = ...
best_model = ...

sample_predictions = best_model.predict(X_test.head(3))

print("Best run:")
print(best_row)
print("Model URI:", best_model_uri)
print("Sample predictions:", sample_predictions)


## Apartado 6: inspeccion con MlflowClient

Recupera el experimento y el mejor run usando `MlflowClient`. Despues lista los artifacts registrados en la raiz del run.

In [ ]:
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment ID:", experiment.experiment_id if experiment else "not found")

# TODO (13): recuperar el run ganador
best_run = ...

# TODO (14): listar los artifacts en la raiz del run
artifacts = ...

print("Best run params:", best_run.data.params)
print("Best run metrics:", best_run.data.metrics)
print("Artifacts:", [artifact.path for artifact in artifacts])


## Preguntas teóricas

1. ¿Qué ventaja aporta usar nested runs frente a registrar todos los modelos candidatos en un único run?
2. ¿En qué se diferencian parámetros, métricas, tags y artifacts dentro de MLflow?
3. ¿Por qué puede ser más razonable seleccionar el mejor modelo por `rmse` en lugar de por `r2` en este problema?
4. ¿Qué información adicional necesitarias registrar si quisieras auditar completamente un experimento meses después?
5. ¿Qué limitaciones prácticas tiene usar un tracking local basado en ficheros frente a un tracking server compartido?


## Extensiones propuestas

1. Registrar un `leaderboard.csv` como artifact del run padre.
2. Añadir una cuarta familia de modelo y justificar si mejora o no al resto.
3. Registrar un alias o nombre estable del mejor modelo usando Model Registry en un servidor MLflow con backend SQL.
